# Notebook 01 — Generate Distillation Data

**Member responsible: A**

Generates teacher CoT traces (D1) and filtered traces (D2) via the Groq API.

Steps:
1. Mount Google Drive + redirect HF cache to Drive (persistent across sessions)
2. Clone repo and install dependencies
3. Set Groq API key
4. Generate raw traces — `generate_traces.py`
5. Filter traces — `filter_traces.py`
6. Build HuggingFace datasets — `build_dataset.py`

In [ ]:
# Step 1a: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 1b: Clone repo
# Paste your GitHub Personal Access Token below (never commit this notebook with the token filled in)
# Generate one at: GitHub > Settings > Developer settings > Personal access tokens > Tokens (classic) > repo scope
import os
token = 'ghp_xxxxxxxxxxxx'  # TODO: paste your token here each session
if not os.path.exists('/content/SC4001-Group133'):
    os.system(f'git clone https://{token}@github.com/TIFFMUN/SC4001-Group133.git /content/SC4001-Group133')
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Step 1c: Redirect HF cache + data/ + results/ to Drive
# MUST run before any HuggingFace imports so datasets/models cache to Drive
# and persist across sessions — no re-downloading needed next time
import sys
sys.path.insert(0, '/content/SC4001-Group133')
from src.setup_cache import setup
setup()   # TODO: if your Drive folder is not MyDrive/SC4001, edit DRIVE_ROOT in setup_cache.py

In [ ]:
# Step 2: Set Groq API key
# TODO: paste your key — do NOT commit this notebook with the key filled in
import os
os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'

In [ ]:
# Step 3 (optional): Quick sanity check — generate 10 traces only
# Edit configs/distillation.yaml: set max_samples: 10
# Then run this cell. Check data/traces/raw_traces.jsonl looks correct before full run.
# Remember to set max_samples: null before the full run.
from src.distillation.generate_traces import generate_traces
generate_traces()

In [ ]:
# Step 4: Filter traces (D2) — keep only traces with correct final answer
# Logs the filter rate (% kept) — record this number for the report
from src.distillation.filter_traces import filter_traces
filter_traces()

In [ ]:
# Step 5a: Build D1 dataset (all traces)
from src.distillation.build_dataset import build_dataset
build_dataset(use_filtered=False)
print('D1 dataset built.')

In [ ]:
# Step 5b: Build D2 dataset (filtered traces only)
build_dataset(use_filtered=True)
print('D2 dataset built.')

In [ ]:
# Step 6: Verify datasets are on Drive
# Because setup() symlinked data/ → Drive, everything already saved there automatically.
# This cell just confirms the files exist.
import os
drive_data = '/content/drive/MyDrive/SC4001/data'
for root, dirs, files in os.walk(drive_data):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f'{path}  ({size_mb:.1f} MB)')